# 02 — Text Preprocessing Pipeline
**ResumeAI Project** | Cleaning and preparing resume and JD text for NLP

## 2.1 Import Libraries

In [ ]:
import re
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Try to import NLP libraries
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    SPACY_OK = True
    print("spaCy loaded successfully!")
except:
    SPACY_OK = False
    print("spaCy not found. Run: pip install spacy && python -m spacy download en_core_web_sm")

try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOPWORDS = set(ENGLISH_STOP_WORDS)
    print("sklearn stopwords loaded!")
except:
    STOPWORDS = {'the','a','an','is','in','at','of','and','to','for','with','on','are','was','be'}
    print("Using basic stopwords fallback")

## 2.2 Sample Resume and JD Texts

In [ ]:
raw_resume = """
John Smith
Email: john.smith@email.com | Phone: +1-555-0123 | LinkedIn: linkedin.com/in/johnsmith

PROFESSIONAL SUMMARY
Experienced software developer with 5+ years of experience building scalable web applications.
Strong background in Python, JavaScript, and cloud technologies (AWS, GCP).

WORK EXPERIENCE
Senior Software Developer - TechCorp Inc. (2020 - Present)
• Developed and maintained React.js frontend applications serving 50,000+ daily users
• Built RESTful APIs using Python/FastAPI reducing response time by 40%
• Led migration of legacy monolith to microservices architecture on AWS
• Mentored team of 3 junior developers, improving team velocity by 25%

Software Developer - StartupXYZ (2018 - 2020)
• Built full-stack features using Django and Vue.js
• Implemented CI/CD pipelines using Jenkins and Docker
• Optimized database queries reducing load time by 60%

EDUCATION
B.S. Computer Science - State University (2018)
GPA: 3.7/4.0

SKILLS
Python, JavaScript, React.js, Node.js, FastAPI, Django, AWS, Docker, Kubernetes,
PostgreSQL, MongoDB, Git, CI/CD, Agile/Scrum, REST APIs, Microservices
"""

raw_jd = """
Senior Software Engineer - FinTech Solutions Ltd

We are looking for a Senior Software Engineer with 4+ years of experience to join our
growing engineering team. You will build scalable financial technology solutions.

Requirements:
- 4+ years of software development experience
- Strong proficiency in Python and JavaScript
- Experience with React.js or Vue.js for frontend development
- Knowledge of cloud platforms (AWS preferred, GCP acceptable)
- Experience with Docker and Kubernetes for containerization
- Familiarity with CI/CD pipelines and DevOps practices
- Experience with RESTful API design and microservices
- Strong knowledge of SQL and NoSQL databases

Nice to have:
- Experience in fintech or banking domain
- Knowledge of machine learning frameworks
- Contributions to open-source projects

We offer competitive salary, remote work options, and excellent benefits.
"""

print("Sample resume length:", len(raw_resume.split()), "words")
print("Sample JD length:", len(raw_jd.split()), "words")

## 2.3 Step-by-Step Preprocessing

In [ ]:
def step1_clean_text(text):
    """Remove emails, URLs, phone numbers, special characters"""
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove phone numbers
    text = re.sub(r'[\+]?[(]?[0-9]{1,4}[)]?[-\s\.]?[(]?[0-9]{1,4}[)]?[-\s\.]?[0-9]{4,9}', '', text)
    # Remove special characters but keep letters, numbers, spaces
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def step2_lowercase(text):
    """Convert to lowercase"""
    return text.lower()

def step3_tokenize(text):
    """Split into word tokens"""
    return text.split()

def step4_remove_stopwords(tokens):
    """Remove common stopwords"""
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]

def step5_lemmatize(tokens):
    """Reduce words to base form using spaCy"""
    if not SPACY_OK:
        return tokens
    text = ' '.join(tokens)
    doc = nlp(text)
    return [token.lemma_ for token in doc if not token.is_stop and len(token.text) > 2]

def step6_extract_noun_phrases(text):
    """Extract meaningful multi-word phrases"""
    if not SPACY_OK:
        return []
    doc = nlp(text[:5000])
    return [chunk.text.lower() for chunk in doc.noun_chunks if len(chunk.text.split()) > 1]

# Apply pipeline
print("PREPROCESSING PIPELINE DEMONSTRATION")
print("=" * 55)

original = raw_resume[:300]
print(f"ORIGINAL (first 300 chars):\n{original}")

s1 = step1_clean_text(raw_resume)
print(f"\nAFTER STEP 1 (Clean):\n{s1[:200]}...")

s2 = step2_lowercase(s1)
print(f"\nAFTER STEP 2 (Lowercase):\n{s2[:200]}...")

s3 = step3_tokenize(s2)
print(f"\nAFTER STEP 3 (Tokenize): {len(s3)} tokens")
print(f"First 20 tokens: {s3[:20]}")

s4 = step4_remove_stopwords(s3)
print(f"\nAFTER STEP 4 (Remove Stopwords): {len(s4)} tokens (removed {len(s3)-len(s4)})")
print(f"First 20 tokens: {s4[:20]}")

s5 = step5_lemmatize(s4)
print(f"\nAFTER STEP 5 (Lemmatize): {len(s5)} tokens")
print(f"First 20 tokens: {s5[:20]}")

## 2.4 Before vs After Comparison

In [ ]:
def full_pipeline(text):
    """Run the complete preprocessing pipeline"""
    cleaned = step1_clean_text(text)
    lowered = step2_lowercase(cleaned)
    tokens  = step3_tokenize(lowered)
    no_stop = step4_remove_stopwords(tokens)
    lemmas  = step5_lemmatize(no_stop)
    return lemmas

resume_processed = full_pipeline(raw_resume)
jd_processed     = full_pipeline(raw_jd)

print("BEFORE vs AFTER PREPROCESSING")
print("-" * 50)
print(f"Resume  — Before: {len(raw_resume.split())} words  |  After: {len(resume_processed)} tokens")
print(f"JD      — Before: {len(raw_jd.split())} words     |  After: {len(jd_processed)} tokens")
print(f"\nReduction rate (Resume): {(1 - len(resume_processed)/len(raw_resume.split()))*100:.1f}%")
print(f"Reduction rate (JD):     {(1 - len(jd_processed)/len(raw_jd.split()))*100:.1f}%")

print(f"\nTop 15 Resume tokens (after preprocessing):")
resume_freq = Counter(resume_processed).most_common(15)
for word, count in resume_freq:
    print(f"  {word}: {count}")

In [ ]:
# Visualize token frequency
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Token Frequency After Preprocessing', fontsize=13, fontweight='bold')

# Resume tokens
res_words, res_counts = zip(*Counter(resume_processed).most_common(12))
axes[0].barh(list(res_words)[::-1], list(res_counts)[::-1], color='#6c63ff', alpha=0.85)
axes[0].set_title('Resume — Top Keywords')
axes[0].set_xlabel('Frequency')

# JD tokens
jd_words, jd_counts = zip(*Counter(jd_processed).most_common(12))
axes[1].barh(list(jd_words)[::-1], list(jd_counts)[::-1], color='#22c55e', alpha=0.85)
axes[1].set_title('Job Description — Top Keywords')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('02_preprocessing.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Noun Phrase Extraction

In [ ]:
if SPACY_OK:
    resume_phrases = step6_extract_noun_phrases(raw_resume)
    jd_phrases     = step6_extract_noun_phrases(raw_jd)

    print("NOUN PHRASES EXTRACTED FROM RESUME:")
    for p in resume_phrases[:15]:
        print(f"  → {p}")

    print("\nNOUN PHRASES EXTRACTED FROM JD:")
    for p in jd_phrases[:15]:
        print(f"  → {p}")

    matched = set(resume_phrases) & set(jd_phrases)
    print(f"\nMATCHED NOUN PHRASES (appear in both):")
    for p in list(matched)[:10]:
        print(f"  ✓ {p}")
else:
    print("Install spaCy to see noun phrase extraction: pip install spacy")
    print("Then: python -m spacy download en_core_web_sm")

## 2.6 Summary

In [ ]:
print("PREPROCESSING PIPELINE SUMMARY")
print("=" * 50)
steps = [
    ("Step 1: Text Cleaning",      "Remove emails, URLs, phone numbers, special chars"),
    ("Step 2: Lowercasing",        "Normalize all text to lowercase"),
    ("Step 3: Tokenization",       "Split into individual word tokens"),
    ("Step 4: Stopword Removal",   "Remove common English stopwords"),
    ("Step 5: Lemmatization",      "Reduce words to base form (spaCy)"),
    ("Step 6: Noun Phrase Chunking","Extract multi-word keyword phrases"),
]
for step, desc in steps:
    print(f"  {step}")
    print(f"    → {desc}")
    print()
print("Pipeline produces clean, normalized tokens ready for TF-IDF and BERT models.")